## Loading the data

In [7]:
import pandas as pd
import numpy as np

In [8]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
geoloc = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

### Completeness Check (Missing Values)

In [ ]:
def missing_values_report(df, table_name):
    total = len(df)                                                 # total number of rows in the data
    missing_values = df.isnull().sum()                              # calculating missing values
    missing_per = ((missing_values/total)*100).round(2)             # calculating missing values in percentage

    report = pd.DataFrame({                                         # storing information in dataframe
        'Column': missing_values.index,
        'Missing Count' : missing_values.values,
        'Missing %' : missing_per.values,
        'Data Type' : df.dtypes.values
    })

    report = report[report['Missing Count'] > 0]                    # isolating the missing values data
    report['Severity'] = report['Missing %'].apply(                 # calculating severity
        lambda x: 'Critical' if x > 20
        else 'Moderate' if x > 5
        else 'Low'
    )
    print(f"\n                      =======Missing Value Report: {table_name} ======= ")
    print(report.to_string(index=False))
    return report


missing_values_report(orders, "orders")


                      =======Missing Value Report: orders ======= 
                       Column  Missing Count  Missing % Data Type Severity
            order_approved_at            160       0.16    object      Low
 order_delivered_carrier_date           1783       1.79    object      Low
order_delivered_customer_date           2965       2.98    object      Low


,Column,Missing Count,Missing %,Data Type,Severity
4,order_approved_at,160,0.16,object,Low
5,order_delivered_carrier_date,1783,1.79,object,Low
6,order_delivered_customer_date,2965,2.98,object,Low


### Duplicates Check

In [39]:
def duplicate_report(df, key_col, table_name):
    total = len(df)
    duplicate_rows = df.duplicated().sum()
    duplicate_keys = df[key_col].duplicated().sum()

    print(f"\n              ======Duplicate Report: {table_name}======")
    print(f"Total Rows              :{total}")
    print(f"Fully Duplicate Rows    :{duplicate_rows} ({round(duplicate_rows*100/total, 2)})")
        


    return duplicate_report

duplicate_report(orders, "order_id", "Orders")


              ======Duplicate Report: Orders======
Total Rows              :99441
Fully Duplicate Rows    :0 (0.0)


<function __main__.duplicate_report(df, key_col, table_name)>

## Validity Check (Format & Range)

In [ ]:
def validity_report(df, table_name):
    print(f"\n          ======Validity Report: {table_name} =   ====")

    # date columns - are they parsable?
    date_cols =[col for col in df.columns if'date' in col or 'timestamp' in col]
    
    for col in date_cols:
        try:
            converted_cols = pd.date_time(df[col],errors = 'coerce')
            failed = converted_cols.isnull().sum() - df[col].isnull().sum()
            print(f"Date column '{col}' : {failed} unparseable values")
        except:
            print(f"Date column '{col}' : Could not convert")

    # Numberic columns - check for negatives or outliers

    num_cols = df.select_dtypes(include = [np.number]).columns
    for col in num_cols:
        negatives = (df[col] < 0).sum()
        if negatives > 0:
            print(f"Numberic columns '{col}': {negatives} negative values")
    

validity_report(orders, "Orders")